In [1]:
import pandas as pd
import geopandas as gpd

# Elementary schools

In [6]:
elem = pd.read_excel(
    '~/Desktop/Etobicoke open close dates - geocoded.xlsx',
    sheet_name='Elem schools close only master'
).dropna(subset=['lat', 'lon']).rename(
    columns={
        'School Name': 'name'
    }
).assign(
    text=lambda df_: (
        df_.Address
            + '<br>Opened: ' + df_['Year opened (at beginning)']
            + '<br>Closed: ' + df_['Year closed (at end)']
    ),
    category=lambda df_: 'elementary_' + df_['Year closed (at end)'].apply(
        lambda x: 'open' if x == '--' else 'closed'
    ),
    name=lambda df_: df_.apply(
        lambda row: row['name'] + ', cl. ' + str(row['Year closed (at end)']) if row['Year closed (at end)'] != '--' else row['name'],
        axis=1
    )
).filter([
    'name', 'category', 'lat', 'lon', 'text'
])

In [7]:
elem_gdf = gpd.GeoDataFrame(
    elem,
    geometry=gpd.points_from_xy(
        elem.lon, elem.lat
    )
).drop(columns=['lat', 'lon']).set_crs(4326)

In [8]:
secondary = pd.read_excel(
    '~/Desktop/Etobicoke open close dates - geocoded.xlsx',
    sheet_name='Secondary Schools-master'
).dropna(subset=['lat', 'lon']).rename(
    columns={
        'School Name': 'name'
    }
).assign(
    text=lambda df_: (
        df_.Address
            + '<br>Opened: ' + df_['Year opened (at beginning)']
            + '<br>Closed: ' + df_['Year closed (at end)']
    ),
    category=lambda df_: 'secondary_' + df_['Year closed (at end)'].apply(
        lambda x: 'open' if x == '--' else 'closed'
    ),
    name=lambda df_: df_.apply(
        lambda row: row['name'] + ', cl. ' + str(row['Year closed (at end)']) if row['Year closed (at end)'] != '--' else row['name'],
        axis=1
    )
).filter([
    'name', 'category', 'lat', 'lon', 'text'
])

In [9]:
secondary_gdf = gpd.GeoDataFrame(
    secondary,
    geometry=gpd.points_from_xy(
        secondary.lon, secondary.lat
    )
).drop(columns=['lat', 'lon']).set_crs(4326)

In [10]:
pd.concat([elem_gdf, secondary_gdf]).to_file('map/overlays/schools.geojson')